In [ ]:
# !pip install langchain
#!pip install -U langchain-classic

In [4]:
# langchain 1버전 이상 패키지에서는 아래 기능은 레거시로 langchain-classsic 패키지로 분리됐다고 함.
#from langchain_core.output_parsers import ResponseSchema, StructuredOutputParser

from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv()
logging.langsmith("CH03-OutputParser")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH03-OutputParser


In [5]:
response_schemas = [
    ResponseSchema(name="answer", description="사용자의 질문에 대한 답변"),
    ResponseSchema(
        name="source",
        description="사용자의 질문에 답하기 위해 사용된 '출처', '웹사이트 주소'이어야 합니다."
    )
]

In [6]:
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

In [7]:
print(output_parser.get_format_instructions())

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"answer": string  // 사용자의 질문에 대한 답변
	"source": string  // 사용자의 질문에 답하기 위해 사용된 '출처', '웹사이트 주소'이어야 합니다.
}
```


In [8]:
format_instructions = output_parser.get_format_instructions()
prompt = PromptTemplate(
    template="answer the users question as best as possible. \n{format_instructions}\n{question}",
    input_variables=["question"],
    partial_variables={"format_instructions": format_instructions},
)


In [9]:
model = ChatOpenAI(temperature=0)
chain = prompt | model | output_parser

In [10]:
chain.invoke({"question": "대한민국의 수도는 어디인가요?"})

{'answer': '대한민국의 수도는 서울입니다.',
 'source': 'https://ko.wikipedia.org/wiki/%EC%84%9C%EC%9A%B8'}